In [5]:
wget https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1846363467.py, line 1)

In [7]:
!uv add 'markitdown[pdf]'

Resolved 136 packages in 15ms
Checked 132 packages in 1.02s


In [9]:
# Downloading the Books
import csv
import io
import shutil
from pathlib import Path
from urllib.error import URLError
from urllib.parse import unquote, urlsplit
from urllib.request import Request, urlopen

CSV_URL = (
    "https://raw.githubusercontent.com/alexeygrigorev/"
    "ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"
)
OUTPUT_DIR = Path("books")


def fetch(url):
    request = Request(url, headers={"User-Agent": "BookDownloader/1.0"})
    return urlopen(request, timeout=60)


def main():
    with fetch(CSV_URL) as response:
        csv_text = response.read().decode("utf-8-sig")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    failures = 0

    for row in csv.DictReader(io.StringIO(csv_text)):
        url = row["pdf_url"].strip()
        filename = Path(unquote(urlsplit(url).path)).name
        destination = OUTPUT_DIR / filename
        temporary = destination.with_suffix(destination.suffix + ".part")

        try:
            print(f"Downloading: {row['title']}")
            with fetch(url) as response, temporary.open("wb") as output:
                shutil.copyfileobj(response, output)

            # Publish the final file only after the download completes.
            temporary.replace(destination)
            print(f"Saved: {destination}")
        except (OSError, URLError) as error:
            temporary.unlink(missing_ok=True)
            failures += 1
            print(f"Failed: {url}\n  {error}")

    return 1 if failures else 0


if __name__ == "__main__":
    raise SystemExit(main())

Downloading: Think Python 2e
Saved: books/thinkpython2.pdf
Downloading: Think DSP
Saved: books/thinkdsp.pdf
Downloading: Think Complexity 2e
Saved: books/thinkcomplexity2.pdf
Downloading: Think Java 2e
Saved: books/thinkjava2.pdf
Downloading: Physical Modeling in MATLAB
Saved: books/PhysicalModelingInMatlab4.pdf
Downloading: Think OS
Saved: books/thinkos.pdf
Downloading: Think C++
Saved: books/Think-C.pdf


SystemExit: 0

/workspaces/ai-engineering-buildcamp-codespaces/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Q1: convert various document formats including PDF to markdown or text format.
Convert all the downloaded PDFs to markdown files and save them to a books_text/ directory.

In [10]:
from pathlib import Path
from markitdown import MarkItDown


def main():
    input_dir = Path("books")
    output_dir = Path("books_text")

    pdf_files = sorted(
        path for path in input_dir.glob("*")
        if path.is_file() and path.suffix.lower() == ".pdf"
    )

    if not pdf_files:
        print(f"No PDFs found in {input_dir.resolve()}")
        return

    output_dir.mkdir(parents=True, exist_ok=True)
    converter = MarkItDown(enable_plugins=False)
    converted = 0

    for pdf_path in pdf_files:
        output_path = output_dir / f"{pdf_path.stem}.md"

        try:
            print(f"Converting: {pdf_path.name}", flush=True)
            result = converter.convert_local(str(pdf_path))
            output_path.write_text(result.markdown, encoding="utf-8")
            converted += 1
            print(f"Saved: {output_path}")
        except Exception as error:
            print(f"Failed: {pdf_path.name}\n  {error}")

    print(
        f"\nConverted {converted}/{len(pdf_files)} PDFs. "
        f"Output: {output_dir.resolve()}"
    )


if __name__ == "__main__":
    main()

2026-10-10 16:43:21.786399685 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Converting: PhysicalModelingInMatlab4.pdf
Saved: books_text/PhysicalModelingInMatlab4.md
Converting: Think-C.pdf
Saved: books_text/Think-C.md
Converting: thinkcomplexity2.pdf
Saved: books_text/thinkcomplexity2.md
Converting: thinkdsp.pdf
Saved: books_text/thinkdsp.md
Converting: thinkjava2.pdf
Saved: books_text/thinkjava2.md
Converting: thinkos.pdf
Saved: books_text/thinkos.md
Converting: thinkpython2.pdf
Saved: books_text/thinkpython2.md

Converted 7/7 PDFs. Output: /workspaces/ai-engineering-buildcamp-codespaces/01-foundation/02-rag/books_text


In [11]:
# How many lines are in the extracted content from the "Think Python" book (2nd edition)?
from pathlib import Path

book_path = Path("books_text/thinkpython2.md")
content = book_path.read_text(encoding="utf-8")

print(f"Number of lines (including blank lines): {len(content.splitlines())}")
# len(content.splitlines()) counts the lines in Python, including blank lines

Number of lines (including blank lines): 9703


In [12]:
len(content) # counts the characters in the Python string

492134

In [ ]:
# wc -l is a terminal command that counts newline characters in a file
!wc -l books_text/thinkpython2.md 

9702 books_text/thinkpython2.md


In [13]:
# Question 2. Chunking for RAG
# task1: prepare your documents:
# 1.Read each markdown file from your books_text/ directory
# 2.Split the content into lines
# 3.Remove empty lines and lines that contain only whitespace
# 4.Turn each book into a dictionary with source (filename) and a content (list of non-empty lines)

from pathlib import Path

documents = []

for book_path in sorted(Path("books_text").glob("*.md")):
    lines = book_path.read_text(encoding="utf-8").splitlines()  # Reads the file as UTF-8 text, then splits it into a list of lines. The newline characters are removed, but empty lines are still included.

    documents.append({
        "source": book_path.name,  # Stores just the filename under the key "source"
        "content": [line for line in lines if line.strip()],  # Creates a list containing only lines with some non-whitespace content, exlude empty lines
    })  # Starts creating a dictionary and adding it to documents

print(f"Prepared {len(documents)} documents.")
# for line in lines examines each line.
# line.strip() removes leading and trailing whitespace for the check.
# if line.strip() excludes lines that become empty.
# The original line is kept, preserving indentation and other whitespace on non-empty lines.

Prepared 7 documents.


In [ ]:
documents[6]

{'source': 'thinkpython2.md',
 'content': ['|     |               | Think      | Python    |',
  '| --- | ------------- | ---------- | --------- |',
  '| How | to Think Like | a Computer | Scientist |',
  '2ndEdition,Version2.4.0',
  '|     |               | Think      | Python    |',
  '| --- | ------------- | ---------- | --------- |',
  '| How | to Think Like | a Computer | Scientist |',
  '2ndEdition,Version2.4.0',
  '|     |     | Allen | Downey    |',
  '| --- | --- | ----- | --------- |',
  '|     |     | Green | Tea Press |',
  'Needham,Massachusetts',
  'Copyright©2015AllenDowney.',
  'GreenTeaPress',
  '9WashburnAve',
  'NeedhamMA02492',
  'Permission is granted to copy, distribute, and/or modify this document under the terms of the',
  'CreativeCommonsAttribution-NonCommercial3.0UnportedLicense,whichisavailableathttp:',
  '//creativecommons.org/licenses/by-nc/3.0/.',
  'TheoriginalformofthisbookisLATEXsourcecode.CompilingthisLATEXsourcehastheeffectofgen-',
  'eratingadevice-

In [14]:
# task2: chunk it
# Use the gitsource package which provides the chunk_documents function
! uv add gitsource

Resolved 136 packages in 1ms
Checked 132 packages in 286ms


In [15]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=100, step=50)

In [16]:
# how to find the chuncks from a specific document for example documents[6]
# each chunk in document_chunks has a "source" field, filter by the filename of documents[6] (the seventh document):

document_chunks[6]

source = documents[6]["source"]

matching_chunks = [
    chunk for chunk in document_chunks
    if chunk["source"] == source
]

print(f"Found {len(matching_chunks)} chunks from {source}")

# Display the matching chunks in your notebook
matching_chunks

Found 189 chunks from thinkpython2.md


[{'start': 0,
  'content': ['|     |               | Think      | Python    |',
   '| --- | ------------- | ---------- | --------- |',
   '| How | to Think Like | a Computer | Scientist |',
   '2ndEdition,Version2.4.0',
   '|     |               | Think      | Python    |',
   '| --- | ------------- | ---------- | --------- |',
   '| How | to Think Like | a Computer | Scientist |',
   '2ndEdition,Version2.4.0',
   '|     |     | Allen | Downey    |',
   '| --- | --- | ----- | --------- |',
   '|     |     | Green | Tea Press |',
   'Needham,Massachusetts',
   'Copyright©2015AllenDowney.',
   'GreenTeaPress',
   '9WashburnAve',
   'NeedhamMA02492',
   'Permission is granted to copy, distribute, and/or modify this document under the terms of the',
   'CreativeCommonsAttribution-NonCommercial3.0UnportedLicense,whichisavailableathttp:',
   '//creativecommons.org/licenses/by-nc/3.0/.',
   'TheoriginalformofthisbookisLATEXsourcecode.CompilingthisLATEXsourcehastheeffectofgen-',
   'eratingade

In [ ]:
source = documents[6]["source"]
source

'thinkpython2.md'

In [17]:
# Question 3. Indexing with minsearch
# use packge `minsearch`
! uv add minsearch

Resolved 136 packages in 1ms
Checked 132 packages in 1ms


In [ ]:
print(type(documents))
print(type(document_chunks))

<class 'list'>
<class 'list'>


In [ ]:
from minsearch import Index

for chunk in document_chunks:
    if isinstance(chunk["content"], list):
        chunk["content"] = "\n".join(chunk["content"])
# document_chunks remains a list of dictionaries, but each chunk’s "content" is now a string.

documents = document_chunks
# here you need to turn the lists into strings
# e.g. with content = "\n".join(chunk["content"])

index = Index(text_fields=["content"])
index.fit(documents)
# This builds the index using the dictionaries in documents. Each dictionary becomes one searchable item—in your case, one chunk. Its "content" must be a string.

Minsearch uses TF-IDF to represent the text numerically, giving words weights based on how frequently they appear within a document and across the collection. These representations let it compare a search query with your chunks.

In [22]:
# How many documents (chunks) did you index?
len(document_chunks)
print(f"Total indexed chunks: {len(documents)}")

Total indexed chunks: 988


In [23]:
# Question 4. Searching and RAG
# Search our index
results = index.search("python function definition", num_results=5)
results[0]

{'start': 1800,
 'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables in other workspaces, even if

In [ ]:
# Question 5. Full RAG
import json
# need to first define the openAI client!
from openai import OpenAI
openai_client = OpenAI()

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-6-luna'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
    }

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [30]:
result = rag("python function definition")

print(result["answer"])
print()
print(f"Input tokens: {result['input_tokens']}")
print(f"Output tokens: {result['output_tokens']}")
print(f"Total tokens: {result['total_tokens']}")

In Python, define a function with `def`, followed by its name, parameters, and an indented body. Use `return` to send a result back to the caller:

```python
def sum_squares(a, b):
    """Return the sum of the squares of a and b."""
    return a**2 + b**2

result = sum_squares(3, 4)
print(result)  # 25
```

Here, `a` and `b` are parameters; `3` and `4` are arguments supplied when the function is called. Variables created inside a function are generally local to it.

Input tokens: 7925
Output tokens: 197
Total tokens: 8122


In [32]:
# Question 6. Structured vs Unstructured Output
# use structured outputs with a Pydantic model

# Define the response model:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [34]:
# Modify the llm and rag functions from Question 5 to use structured outputs
import json
# need to first define the openAI client!
from openai import OpenAI
openai_client = OpenAI()

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm_structured(user_prompt, output_type, instructions, model='gpt-6-luna'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return {
            "answer": response.output_text,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "total_tokens": response.usage.total_tokens,
        }

def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )